# 04 — Game Similarity Graph

We build a graph where each node is a game and each edge connects a game to one of its 10 most similar games. The edge weight is the Jaccard similarity.

In this notebook, we build a graph where each node is a game and each edge connects a game to one of its 10 most similar games. The edge weight is the Jaccard similarity.

In [1]:
from pathlib import Path

import networkx as nx
import pandas as pd


candidates = pd.read_parquet("../data/processed/game_similarity_candidates.parquet")

# Count each pair in both directions, so every game sees all its neighbours.
both_directions = pd.concat([
    candidates,
    candidates.rename(columns={"game_a": "game_b", "game_b": "game_a"})
])

edges = (
    both_directions
    .sort_values("jaccard_similarity", ascending=False)
    .groupby("game_a")
    .head(10)
)

G = nx.from_pandas_edgelist(edges, "game_a", "game_b", edge_attr="jaccard_similarity")

print("Games:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Games: 2363
Edges: 18074


In [2]:
OUTPUT_PATH = Path("../data/processed/game_graph_edges.parquet")

nx.to_pandas_edgelist(G, source="game_a", target="game_b").to_parquet(OUTPUT_PATH, index=False)

print("Saved graph to:", OUTPUT_PATH)

Saved graph to: ../data/processed/game_graph_edges.parquet
